# 05 - Evaluate LLaMEA Champions (N=10 Independent Runs)

This notebook:
1. Loads problem-specific **Clean** and **Noisy** champions from `data/champions.json` (generated by Notebook 04).
2. Executes each champion **N=10 independent times** on target BBOB problems across multiple dimensions and noise levels.
3. Evaluates Clean Champions on clean settings (`noise_std = 0.0`) and Noisy Champions on noisy settings (`noise_std > 0.0`).
4. Attaches IOH Analyzer via `problem.attach_analyzer(...)` to output IOH `.dat` performance files to `data/ioh_logs/{dim}D/std_{noise_std}/f{p_id}/llamea_champion_{mode}/`.

In [1]:
import sys
import json
import numpy as np
from pathlib import Path

# Add src to path
cwd = Path('.').resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from infra.problems.bbob import BBOBProblem
from domain.services.noise_strategy import MultiplicativeNoiseStrategy, NoNoiseStrategy
from synthesis.execution import AlgorithmExecutor

# ── Experiment Configuration ─────────────────────────────
CHAMPIONS_PATH  = PROJECT_ROOT / 'data' / 'champions.json'
IOH_LOGS_DIR    = PROJECT_ROOT / 'data' / 'ioh_logs'
DIMS            = [2, 3]             # Dimensions to evaluate
N_RUNS          = 10                 # Independent runs per config
BUDGET          = 100000             # Function evaluation budget
TIMEOUT_SECONDS = 30.0
# ─────────────────────────────────────────────────────────

print(f'Champions JSON: {CHAMPIONS_PATH}')
print(f'IOH Logs Output: {IOH_LOGS_DIR}')
print(f'Target Dimensions: {DIMS}')
print(f'Runs per champion: {N_RUNS}')
print(f'Budget: {BUDGET} evaluations')

## 1. Load Champions JSON

In [2]:
if not CHAMPIONS_PATH.exists():
    raise FileNotFoundError(f'Champions file not found at {CHAMPIONS_PATH}. Please run Notebook 04 first.')

with open(CHAMPIONS_PATH, 'r') as f:
    champions = json.load(f)

print(f'Loaded {len(champions)} champion configuration(s):')
for key, info in champions.items():
    mode_str = info.get('mode', 'all').upper()
    print(f"  {key} [{mode_str}]: {info['algorithm_name']} (from Exp #{info['experiment_id']}, std={info['noise_std']})")

## 2. Execute Champion Evaluation Benchmark

In [3]:
executor = AlgorithmExecutor(timeout_seconds=TIMEOUT_SECONDS)

for key, info in champions.items():
    p_id = int(info['problem_id'])
    mode = info.get('mode', 'all')
    champ_noise_std = float(info.get('noise_std', 0.0))
    
    code_file = PROJECT_ROOT / info['code_path'] if not Path(info['code_path']).is_absolute() else Path(info['code_path'])
    if not code_file.exists():
        print(f'[WARN] Code file for {key} not found at {code_file}. Skipping.')
        continue
        
    code_content = code_file.read_text(encoding='utf-8')
    algo_name = info['algorithm_name']
    
    # Select noise levels to test against for this champion
    if mode == 'clean':
        eval_noise_levels = [0.0]
    elif mode == 'noisy':
        eval_noise_levels = [champ_noise_std] if champ_noise_std > 0 else [0.05, 0.1]
    else:
        eval_noise_levels = [0.0, 0.05, 0.1]
        
    for dim in DIMS:
        for noise_std in eval_noise_levels:
            # Standardized directory structure: data/ioh_logs/{dim}D/std_{noise_std}/f{p_id}/
            out_dir = IOH_LOGS_DIR / f'{dim}D' / f'std_{noise_std}' / f'f{p_id}'
            out_dir.mkdir(parents=True, exist_ok=True)
            
            folder_name = f'llamea_champion_{mode}'
            print()
            print(f'=== Evaluating Champion {key} for f{p_id} ({dim}D, noise={noise_std}): {algo_name} (N={N_RUNS} runs, Budget={BUDGET}) ===')
            
            noise_strat = MultiplicativeNoiseStrategy(noise_std) if noise_std > 0.0 else NoNoiseStrategy()
            problem = BBOBProblem(
                problem_id=p_id,
                dim=dim,
                instance_id=1,
                noise_strategy=noise_strat,
            )
            
            # Attach IOH logger once for all N runs
            problem.attach_analyzer(
                log_dir=out_dir,
                folder_name=folder_name,
                algorithm_name=f'LLaMEA {key.capitalize()} Champion',
                algorithm_info=f'dim={dim}, noise_std={noise_std}, exp={info.get("experiment_id", "N/A")}',
            )
            
            clean_errors = []
            for run_idx in range(1, N_RUNS + 1):
                problem.reset()
                try:
                    best_x, best_y = executor.execute_algorithm(
                        code=code_content,
                        name=algo_name,
                        dim=dim,
                        problem=problem.get_objective_fn(),
                        budget=BUDGET,
                    )
                    
                    if best_x is not None:
                        clean_y = problem.eval_clean(best_x)
                        clean_err = abs(clean_y - problem.true_optimum)
                    else:
                        clean_err = float('inf')
                        
                    clean_errors.append(clean_err)
                    print(f'  Run {run_idx:2d}/{N_RUNS}: evals={problem.evaluations}, final clean error={clean_err:.6e}')
                except Exception as exc:
                    print(f'  Run {run_idx:2d}/{N_RUNS} FAILED: {exc}')
                    clean_errors.append(float('inf'))
            
            problem.close_logger()
            valid_errs = [e for e in clean_errors if not np.isinf(e)]
            med_err = np.median(valid_errs) if valid_errs else float('inf')
            print(f'  f{p_id} {mode.upper()} ({dim}D, noise={noise_std}) Champion Median Clean Error across {N_RUNS} runs: {med_err:.6e}')

print('\nChampion evaluations complete!')